# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

import json
from typing import List, Dict, Optional

from dotenv import load_dotenv, find_dotenv
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [4]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
load_dotenv()

True

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# Load the database built in Part 1
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)


@tool
def retrieve_game(query: str) -> List[Dict]:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    - Publisher: Company that published the game
    - Genre: Genre of the game
    - Distance: How far the result is from the query (lower = closer match)
    """
    try:
        results = collection.query(query_texts=[query], n_results=3)
    except Exception as e:
        return [{"error": f"Vector DB search failed: {e}"}]

    games = []
    for metadata, distance in zip(results["metadatas"][0], results["distances"][0]):
        games.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "Publisher": metadata.get("Publisher"),
            "Genre": metadata.get("Genre"),
            "Distance": round(distance, 4),
        })
    return games

#### Evaluate Retrieval Tool

In [7]:
# If your notebook doesn't already define it, add this:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


judge_llm = LLM(model="gpt-4o-mini", temperature=0.0)


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict]) -> EvaluationReport:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not. "
        "Mark useful as true only if the documents directly contain the information "
        "needed (the right game, and the specific fact asked about). "
        "If the game is missing or the documents are about different games, mark it false.\n\n"
        f"QUESTION: {question}\n\n"
        f"DOCUMENTS: {json.dumps(retrieved_docs, indent=2, default=str)}"
    )

    response = judge_llm.invoke(
        input=[
            SystemMessage(content="You are a strict evaluator of retrieved documents."),
            UserMessage(content=prompt),
        ],
        response_format=EvaluationReport,
    )
    return EvaluationReport.model_validate_json(response.content)

#### Game Web Search Tool

In [8]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str) -> List[Dict]:
    """
    Web search: searches the internet for information about the game industry
    when the local vector DB doesn't have enough information.

    args:
    - question: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - title: title of the web page
    - url: source URL (cite this in the final answer)
    - content: relevant text extracted from the page
    """
    try:
        response = tavily_client.search(
            query=question,
            max_results=3,
            include_answer=False,
        )
    except Exception as e:
        return [{"error": f"Web search failed: {e}"}]

    return [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
        }
        for r in response.get("results", [])
    ]

### Agent

In [9]:
instructions = """
You are UdaPlay, an AI research assistant for the video game industry.
Answer questions about games: titles, release dates, platforms, genres,
publishers and descriptions.

Follow this process for EVERY question:
1. Call retrieve_game to search the internal vector database first.
2. Call evaluate_retrieval with the original question and the retrieved
   documents to decide whether they are enough to answer.
3. If useful is true, answer from the retrieved documents.
4. If useful is false (the game is missing, or the documents don't contain
   the fact asked about), call game_web_search and answer from those results.
5. Never answer from your own background knowledge. Use only tool results.

Final answer format:
- Give a clear, natural-language answer to the question.
- Source: "Internal database" or the web URL(s) you used. Always include
  the URL when the answer comes from the web.
- Confidence: High, Medium or Low, with one short reason.
- If you combined internal and web information, say which part came from where.

If a question refers to an earlier one ("that game", "the same platform"),
use the conversation history to resolve it.
If neither source has the answer, say so honestly instead of guessing.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

In [11]:
session_id = "udaplay_session_1"

queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

runs = []
for q in queries:
    print("=" * 80)
    print(f"QUESTION: {q}")
    print("=" * 80)
    run = agent.invoke(q, session_id=session_id)
    runs.append(run)

    final_state = run.get_final_state()

    # Show which tools the agent used, in order
    for msg in final_state["messages"]:
        for call in getattr(msg, "tool_calls", None) or []:
            print(f"-> Tool called: {call.function.name}")

    print()
    print(final_state["messages"][-1].content)
    print()

QUESTION: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
-> Tool called: retrieve_game
-> Tool called: evaluate_retrieval
-> Tool called: retrieve_game
-> Tool called: evaluate_retrieval
-> Tool called: retrieve_game
-> Tool called: evaluate_retrieval
-> Tool called: game_web_search

Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon games, introducing new regions, Pokémon, and gameplay mechanics.

Source: "Internal database"  
Confidence: High, because the retrieved document directly provided the release year.

QUESTION: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
-> Tool called

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes